In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

csv_path = '/content/drive/MyDrive/CAPSTONE2026_AI/final_synthetic_ami_data.csv'
model_save_dir = '/content/drive/MyDrive/CAPSTONE2026_AI/saved_models'

# CSV 파일 존재 여부 팩트 체크
if os.path.exists(csv_path):
    print(" CSV 데이터 파일 인식 성공!")
else:
    print(" CSV 파일이 없습니다. 구글 드라이브 경로를 다시 확인하세요.")

# 모델 저장 폴더 생성 (구글 드라이브 내부에 영구 저장되도록 설정)
os.makedirs(model_save_dir, exist_ok=True)
print(f" 모델 저장 경로 세팅 완료: {model_save_dir}")

 CSV 데이터 파일 인식 성공!
 모델 저장 경로 세팅 완료: /content/drive/MyDrive/CAPSTONE2026_AI/saved_models


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import joblib
import os
import random
import copy
from tqdm import tqdm
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# ==========================================
# 하드웨어 결정론적 난수 고정 (Reproducibility)
# ==========================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # cuDNN의 비결정적 연산 차단 (GPU와 CPU의 수학적 연산 결과 동기화)[cite: 1]
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# ==========================================
# 아키텍처 정의 (96-in / 96-out) - 로컬과 100% 동일
# ==========================================
class KHNPSmartDRNet(nn.Module):
    def __init__(self, num_users, user_embed_dim=16, feature_dim=5, hidden_dim=128, num_layers=3, pred_len=96):
        super().__init__()
        self.user_embedding = nn.Embedding(num_embeddings=num_users, embedding_dim=user_embed_dim)
        self.lstm = nn.LSTM(input_size=feature_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=True)
        self.layer_norm1 = nn.LayerNorm(hidden_dim * 2)
        self.attention = nn.MultiheadAttention(embed_dim=hidden_dim * 2, num_heads=8, batch_first=True)
        self.layer_norm2 = nn.LayerNorm(hidden_dim * 2)
        self.fc = nn.Sequential(
            nn.Linear((hidden_dim * 2) + user_embed_dim, 256),
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3), nn.Linear(256, pred_len)
        )
    def forward(self, x, user_id):
        user_vec = self.user_embedding(user_id)
        lstm_out, _ = self.lstm(x)
        lstm_out = self.layer_norm1(lstm_out)
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        attn_out = self.layer_norm2(attn_out + lstm_out)
        combined_features = torch.cat((attn_out[:, -1, :], user_vec), dim=1)
        return self.fc(combined_features)

# ==========================================
# 데이터셋 정의 - 로컬과 100% 동일
# ==========================================
class PowerDataset(Dataset):
    def __init__(self, df, feature_cols, seq_length=96, pred_length=96):
        self.seq_length, self.pred_length = seq_length, pred_length
        self.features = torch.tensor(df[feature_cols].values, dtype=torch.float32)
        self.targets = torch.tensor(df['kwh_scaled'].values, dtype=torch.float32)
        self.user_ids = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.valid_indices = []
        user_groups = df.groupby('user_idx').indices
        for user, indices in user_groups.items():
            start, end = indices[0], indices[-1]
            max_start = end - (seq_length + pred_length) + 1
            if max_start > start:
                self.valid_indices.extend(range(start, max_start))
    def __len__(self): return len(self.valid_indices)
    def __getitem__(self, idx):
        s = self.valid_indices[idx]
        return self.features[s:s+self.seq_length], self.user_ids[s], self.targets[s+self.seq_length:s+self.seq_length+self.pred_length]

# ==========================================
# 조기 종료 방어 기제 (Early Stopping)
# ==========================================
class EarlyStopping:
    def __init__(self, patience=7, delta=0.0005):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_loss = float('inf')
        self.best_model_weights = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.best_model_weights = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            print(f"Validation Loss가 줄어들지 않음 (조기 종료 카운트: {self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.early_stop = True

# ==========================================
# Data Pipeline 동기화
# ==========================================
print("CSV 데이터 로드 및 InfluxDB 모사 처리 중...")
csv_path = '/content/drive/MyDrive/CAPSTONE2026_AI/final_synthetic_ami_data.csv' # 구글 드라이브 경로
df = pd.read_csv(csv_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 로컬 InfluxDB의 aggregateWindow(every: 15m, fn: mean)를 수학적으로 완벽히 모사
# 1분 단위 원본 CSV 데이터를 가구별(user_id)로 묶어 15분 간격의 평균값으로 다운샘플링
print("⚡ 15분 단위 다운샘플링(Resampling) 수행 중...")
df = df.set_index('timestamp').groupby('user_id').resample('15min')['kwh_usage'].mean().reset_index()
df = df.dropna() # 보간 과정 중 생길 수 있는 결측치 제거
df = df.sort_values(by=['user_id', 'timestamp']).reset_index(drop=True)

# 피처 엔지니어링
user_to_idx = {user: idx for idx, user in enumerate(df['user_id'].unique())}
df['user_idx'] = df['user_id'].map(user_to_idx)
df['hour_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.hour / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.hour / 24.0)
df['day_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.dayofweek / 7.0)
df['day_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.dayofweek / 7.0)

split_time = df['timestamp'].quantile(0.8)
train_df, val_df = df[df['timestamp'] < split_time].copy(), df[df['timestamp'] >= split_time].copy()

scaler = MinMaxScaler()
train_df['kwh_scaled'] = scaler.fit_transform(train_df[['kwh_usage']])
val_df['kwh_scaled'] = scaler.transform(val_df[['kwh_usage']])
features = ['kwh_scaled', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']

actual_num_users = len(user_to_idx)
train_loader = DataLoader(PowerDataset(train_df, features), batch_size=256, shuffle=True, drop_last=True, pin_memory=True)
val_loader = DataLoader(PowerDataset(val_df, features), batch_size=256, shuffle=False, pin_memory=True)

# ==========================================
# 지능형 훈련 루프 (Validation & Checkpointing)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KHNPSmartDRNet(num_users=actual_num_users).to(device)
criterion = nn.HuberLoss(delta=1.0)
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)

# 지역 최적해 탈출 스케줄러
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

# 5번 이상 검증 오차가 안 줄면 즉시 훈련을 종료하여 과적합 방지
early_stopping = EarlyStopping(patience=5, delta=0.0005)

print(f" {device} 기반 지능형 훈련 시작!")
num_epochs = 100

for epoch in range(num_epochs):
    # --- 1단계: 훈련 ---
    model.train()
    train_loss = 0.0
    for x, u, y in tqdm(train_loader, desc=f"Epoch {epoch+1} Train", leave=False):
        x, u, y = x.to(device, non_blocking=True), u.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        loss = criterion(model(x, u), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_loader.dataset)

    # --- 2단계: 검증 ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad(): # 검증 시에는 가중치를 업데이트하지 않음
        for x, u, y in tqdm(val_loader, desc=f"Epoch {epoch+1} Val", leave=False):
            x, u, y = x.to(device, non_blocking=True), u.to(device, non_blocking=True), y.to(device, non_blocking=True)
            loss = criterion(model(x, u), y)
            val_loss += loss.item() * x.size(0)
    val_loss /= len(val_loader.dataset)

    scheduler.step()

    print(f"Epoch {epoch+1:03d}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    # --- 3단계: 조기 종료 및 브레이크 판단 ---
    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print(f"[조기 종료] 모델이 과적합을 시작하여 {epoch+1} 에포크에서 훈련을 강제 중단합니다.")
        model.load_state_dict(early_stopping.best_model_weights)
        break

# 4. 모델 저장
save_dir = '/content/drive/MyDrive/CAPSTONE2026_AI/saved_models'
os.makedirs(save_dir, exist_ok=True)
torch.save(model.state_dict(), f'{save_dir}/khnp_dr_best_model.pth')
joblib.dump(scaler, f'{save_dir}/scaler.pkl')
joblib.dump(user_to_idx, f'{save_dir}/user_to_idx.pkl')
print(f"모델 저장 성공! (경로: {save_dir})")

CSV 데이터 로드 및 InfluxDB 모사 처리 중...
⚡ 15분 단위 다운샘플링(Resampling) 수행 중...
🔥 cuda 기반 지능형 훈련 시작!


Epoch 001/100 | Train Loss: 0.0021 | Val Loss: 0.0127 | LR: 0.000488


Epoch 002/100 | Train Loss: 0.0009 | Val Loss: 0.0135 | LR: 0.000452
Validation Loss가 줄어들지 않음 (조기 종료 카운트: 1/5)


Epoch 003/100 | Train Loss: 0.0007 | Val Loss: 0.0140 | LR: 0.000397
Validation Loss가 줄어들지 않음 (조기 종료 카운트: 2/5)


Epoch 004/100 | Train Loss: 0.0007 | Val Loss: 0.0143 | LR: 0.000327
Validation Loss가 줄어들지 않음 (조기 종료 카운트: 3/5)


Epoch 005/100 | Train Loss: 0.0007 | Val Loss: 0.0142 | LR: 0.000250
Validation Loss가 줄어들지 않음 (조기 종료 카운트: 4/5)


Epoch 006/100 | Train Loss: 0.0006 | Val Loss: 0.0140 | LR: 0.000173
Validation Loss가 줄어들지 않음 (조기 종료 카운트: 5/5)
[조기 종료] 모델이 과적합을 시작하여 6 에포크에서 훈련을 강제 중단합니다.
모델 저장 성공! (경로: /content/drive/MyDrive/CAPSTONE2026_AI/saved_models)
